# M07 — Parquet y partición de negocio

[← Anterior](../M06-optimizacion-ejecucion/03-lab-cache-particionado.ipynb) · [Siguiente →](02-lab-parquet-layout.ipynb)

El pipeline acaba en un directorio que otro proceso puede leer mañana. `repartition` baraja **memoria**. `partitionBy` en el `write` organiza **disco**.

Ejecuta las celdas **aquí**, en este mismo fichero. No lo copies a otro sitio.

Kernel: **Python (NovaShop)**.


## Arranque

Ejecuta estas dos celdas. Localizan el repo y dejan una `SparkSession` lista.


In [ ]:
import sys
from pathlib import Path

_here = Path.cwd().resolve()
ROOT = next(
    p
    for p in [_here, *_here.parents]
    if (p / "labs" / "_shared" / "session.py").is_file()
)
sys.path.insert(0, str(ROOT / "labs" / "_shared"))

from paths import RAW, STAGING, CURATED
from session import get_spark

print("ROOT   ", ROOT)
print("RAW    ", RAW, "existe:", RAW.is_dir())
print("STAGING", STAGING)
print("CURATED", CURATED)


In [ ]:
spark = get_spark('novashop-clase-m07')
print(spark.version, spark.sparkContext.master)


## Escribes carpetas, no un Excel


In [ ]:
from pyspark.sql import Row
from pyspark.sql.functions import col

demo = spark.createDataFrame([
    Row(order_id="O1", order_month="2024-01", gmv=10.0),
    Row(order_id="O2", order_month="2024-01", gmv=20.0),
    Row(order_id="O3", order_month="2024-02", gmv=5.0),
])
dest = CURATED / "_demo_sales"
CURATED.mkdir(parents=True, exist_ok=True)
demo.write.mode("overwrite").partitionBy("order_month").parquet(str(dest))
print(sorted(p.name for p in dest.iterdir() if p.is_dir()))
enero = spark.read.parquet(str(dest)).where(col("order_month") == "2024-01")
enero.explain("formatted")
print("enero", enero.count(), "total", spark.read.parquet(str(dest)).count())


**Siguiente:** [lab de parquet](02-lab-parquet-layout.ipynb) sobre el fact real.
